In [1]:
import pandas as pd
import numpy as np
import re

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
print("Total Samples:", len(df))

print("\nClass Distribution:\n", df['sentiment'].value_counts())

print("\nSample Text:")
print(df['review'].iloc[0])

Total Samples: 50000

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Text:
positive


In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"[^\w\s]", "", text)

    tokens = text.split()

    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

In [9]:
df['clean_text'] = df['review'].apply(preprocess_text)

df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch 1 oz episod youll hoo...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [10]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [12]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [13]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, average='weighted'),
        "Recall": recall_score(y_test, preds, average='weighted'),
        "F1 Score": f1_score(y_test, preds, average='weighted')
    }

In [14]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

results_bow = {}

for name, model in models.items():
    results_bow[name] = train_and_evaluate(
        model, X_train_bow, X_test_bow, y_train, y_test
    )

results_bow

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'Logistic Regression': {'Accuracy': 0.873,
  'Precision': 0.8730981692548765,
  'Recall': 0.873,
  'F1 Score': 0.8729814427263576},
 'Naive Bayes': {'Accuracy': 0.8469,
  'Precision': 0.8469913844576175,
  'Recall': 0.8469,
  'F1 Score': 0.8469005588151396},
 'Decision Tree': {'Accuracy': 0.722,
  'Precision': 0.7222649825940216,
  'Recall': 0.722,
  'F1 Score': 0.7219699744434751}}

In [15]:
results_tfidf = {}

for name, model in models.items():
    results_tfidf[name] = train_and_evaluate(
        model, X_train_tfidf, X_test_tfidf, y_train, y_test
    )

results_tfidf

{'Logistic Regression': {'Accuracy': 0.8854,
  'Precision': 0.8857666261905509,
  'Recall': 0.8854,
  'F1 Score': 0.8853563355867297},
 'Naive Bayes': {'Accuracy': 0.8497,
  'Precision': 0.8497471039463977,
  'Recall': 0.8497,
  'F1 Score': 0.8496853952843585},
 'Decision Tree': {'Accuracy': 0.714,
  'Precision': 0.7142536504648469,
  'Recall': 0.714,
  'F1 Score': 0.7139703689480785}}

In [16]:
bow_df = pd.DataFrame(results_bow).T
tfidf_df = pd.DataFrame(results_tfidf).T

print("BoW Results:\n", bow_df)
print("\nTF-IDF Results:\n", tfidf_df)

BoW Results:
                      Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.8730   0.873098  0.8730  0.872981
Naive Bayes            0.8469   0.846991  0.8469  0.846901
Decision Tree          0.7220   0.722265  0.7220  0.721970

TF-IDF Results:
                      Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.8854   0.885767  0.8854  0.885356
Naive Bayes            0.8497   0.849747  0.8497  0.849685
Decision Tree          0.7140   0.714254  0.7140  0.713970


Insights and Analysis

Preprocessing greatly enhanced the performance of the model by removing noise from the data. Lowercase conversion, stopword removal, and stemming helped minimize redundant words, thereby reducing the overall vocabulary of the data. This made it more efficient for learning.

TF-IDF outperformed the Bag of Words model since it focuses more on the importance of words, giving more importance to words that are not frequently occurring but are actually meaningful. 

Among the three models, Logistic Regression showed the highest performance. This is because it works well for linearly separable data, which is the case for text data. Although Naive Bayes showed a good performance, it was not as good as Logistic Regression. The decision trees were not reliable since they were overfitting.